# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [ ]:
!pip install -U tokenizers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [ ]:
# A smaller model that fits on a free Colab GPU
model_name = "ibm-granite/granite-3.3-8b-instruct"

# 2. Hardware optimization (Quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
pip install -U bitsandbytes>=0.46.1

# 3. Load the tokenizer & model

In [ ]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True # Does your model need it?
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?

Loading Tokenizer for ibm-granite/granite-3.3-8b-instruct...


config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    torch_dtype=torch.float16
)

Loading Model on Colab T4 GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [ ]:
# 1. Define the role based promting (Week 2 Prototyping)
messages = [
    {
        "role": "system",
        "content": (
            "You are an expert software architect specializing in the Apache Hadoop ecosystem. "
            "Your task is to analyze Java source code from the Hadoop MapReduce 'Client core' component "
            "and identify its architectural purpose."
        )
    },
    {
        "role": "user",
        "content": """Please analyze the following classes from a single cluster identified during architectural recovery.

        ### Task:
        1. Provide a concise *Architectural Title* for this cluster.
        2. Write a *High-Level Descriptive Summary* (3-4 sentences) explaining how these classes interact to support Hadoop MapReduce Client core functionality.

        ### Source Code to Analyze:
        <source_code>
org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.Job$12
org.apache.hadoop.mapreduce.JobContextz
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.JobSubmitter$1
org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
org.apache.hadoop.mapreduce.TaskCompletionEvent
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.tools.CLI
        </source_code>
        """
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

Generating the optimized architectural analysis...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### Architectural Title:
Hadoop MapReduce Client Core Interaction and Job Management

### High-Level Descriptive Summary:
The provided classes form the core of the Hadoop MapReduce client, facilitating job submission, management, and execution within a Hadoop cluster. The `Job` class encapsulates the details of a MapReduce job, while `JobSubmitter` handles the process of submitting jobs to the cluster. `JobContext` and `JobContextImpl` provide the necessary context for job execution. `ClientProtocol` enables communication between the client and the job tracker. 

The `CryptoUtils` class suggests the involvement of cryptographic operations, possibly for secure communication. `JobResourceUploader` might be responsible for uploading resources required by the job to the cluster. `JobStatus` provides the current status of a job, and `TaskCompletionEvent` could be used to notify when tasks within a job have completed. 

`DelegationTokenSecretManager` and `DelegationTokenSelector` indicate su

In [ ]:
# 1. Define the role based promting (Week 2 Prototyping)
messages = [
    {
       "role": "system",
        "content": (
            "You are a Senior Security Architect specializing in distributed systems and Kerberos authentication. "
            "Your task is to analyze Java source code from the Hadoop MapReduce 'Client core' component "
            "and identify how this cluster manages secure job submission."
        )
    },
    {
        "role": "user",
        "content": """Analyze the following classes for security-critical functions:
        <source_code>

        ### Task:
        1. Provide a concise *Architectural Title* for this cluster.
        2. Write a *High-Level Descriptive Summary* (3-4 sentences) explaining how these classes interact to support Hadoop MapReduce Client core functionality.

        ### Source Code to Analyze:
        <source_code>
org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.Job$12
org.apache.hadoop.mapreduce.JobContextz
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.JobSubmitter$1
org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
org.apache.hadoop.mapreduce.TaskCompletionEvent
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.tools.CLI
        </source_code>
        """
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

Generating the optimized architectural analysis...

### Architectural Title:
Secure Distributed Job Management in Hadoop MapReduce Client Core

### High-Level Descriptive Summary:
The Hadoop MapReduce Client core employs a robust architecture for secure job submission and management. The `JobSubmitter` class is central to this process, interacting with the `ClientProtocol` to send job submissions to the `JobTracker`. The `CryptoUtils` class plays a crucial role in handling cryptographic operations, ensuring secure communication between the client and the cluster. The `Job` class encapsulates the job details, while `JobContext` provides necessary contextual information for job execution. The `DelegationTokenSecretManager` and `DelegationTokenSelector` facilitate the use of delegation tokens for secure authentication, ensuring only authorized users can submit jobs. The `JobResourceUploader` handles the distribution of necessary resources, and `TaskCompletionEvent` monitors job progress. 

In [ ]:
# 1. Define the role based promting (Week 2 Prototyping)
messages = [
    {
       "role": "system",
        "content": (
            "You are a Principal Software Engineer focused on design patterns and system maintainability. "
            "Your task is to analyze the provided Hadoop MapReduce classes and identify structural "
            "patterns (like Factory, Proxy, or Bridge) that facilitate job context management."
        )
    },
    {
        "role": "user",
        "content": """Identify design patterns used in the following classes:
        ### Task:
        1. Provide a concise *Architectural Title* for this cluster.
        2. Write a *High-Level Descriptive Summary* (3-4 sentences) explaining how these classes interact to support Hadoop MapReduce Client core functionality.

        ### Source Code to Analyze:
        <source_code>
org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.Job$12
org.apache.hadoop.mapreduce.JobContextz
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.JobSubmitter$1
org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
org.apache.hadoop.mapreduce.TaskCompletionEvent
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.tools.CLI
        </source_code>
        """
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

Generating the optimized architectural analysis...

### Architectural Title:
Hadoop MapReduce Client Core Framework

### High-Level Descriptive Summary:
The Hadoop MapReduce Client core functionality is orchestrated by a set of interconnected classes. The `Job` class is central, representing a MapReduce job and encapsulating its context (`JobContext` implemented by `JobContextImpl`). The `JobSubmitter` class facilitates job submission, handling job configuration and interaction with the `ClientProtocol` to communicate with the Hadoop Distributed File System (HDFS) and MapReduce runtime. The `CryptoUtils` class ensures secure communication, while `JobResourceUploader` manages the distribution of job resources. The `Job` class also interacts with `TaskCompletionEvent` to monitor job progress. The `DelegationTokenSelector` and `DelegationTokenSecretManager` handle security aspects, particularly for delegation tokens. The `ClientDistributedCacheManager` manages caching of distributed files